# Neighbouring candidates

The purppose of this Jupyter file is to locate potential stations to be merged together in the ceation of a test instance

## Imports

In [135]:
import gzip
import json 
import pandas as pd
import folium
from folium.features import DivIcon
from sklearn.cluster import DBSCAN
import numpy as np
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform

## Read old TD data

In [136]:
# read in the json.gz file from the instances folder
# Current location: policies/sjovik_sund/scripts/test_instance/
# Target: instances/TD_W34_old.json.gz
# Path: go up 4 levels to reach FOMOsim root, then down to instances

with gzip.open('../../../../instances/TD_W34_old.json.gz', 'rt', encoding='utf-8') as f: 
    data = json.load(f)
    
print("Data keys:", data.keys())

# get all stations
stations = data['stations']

# Convert to DataFrame
df_stations = pd.DataFrame(stations)

# Split the 'location' column (which is a list [lat, lon]) into separate columns
df_stations[['latitude', 'longitude']] = pd.DataFrame(df_stations['location'].tolist(), index=df_stations.index)

df_stations.head()

Data keys: dict_keys(['name', 'map', 'map_boundingbox', 'stations', 'bike_class', 'traveltime', 'traveltime_stdev', 'traveltime_vehicle', 'traveltime_vehicle_stdev'])


,id,location,is_depot,capacity,num_bikes,leave_intensities,leave_intensities_stdev,arrive_intensities,arrive_intensities_stdev,move_probabilities,latitude,longitude
0,0,"[63.40772802863199, 10.39705323440262]",False,21,10,"[[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.2, 0.8, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.400000000000...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.8000000...","[[[0.014925373134328358, 0.014925373134328358,...",63.407728,10.397053
1,1,"[63.414746566093925, 10.397386363673064]",False,15,7,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.2, 0.4, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.4000000000000001,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.6, 0.0, 0.2,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.4000000000000001,...","[[[0.014925373134328358, 0.014925373134328358,...",63.414747,10.397386
2,2,"[63.436025749373755, 10.430998463879718]",False,24,12,"[[0.0, 0.0, 0.0, 0.0, 0.2, 1.8, 0.4, 0.2, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.4000000000000001, 2.22...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.400000000000...","[[[0.014925373134328358, 0.014925373134328358,...",63.436026,10.430998
3,3,"[63.42126270770024, 10.386516005020496]",False,18,8,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.6, 0.4, 0.0, 0.4,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.7999999999999999,...","[[0.0, 0.0, 0.0, 0.0, 0.2, 1.4, 1.6, 0.2, 1.0,...","[[0.0, 0.0, 0.0, 0.0, 0.4000000000000001, 1.95...","[[[0.014925373134328358, 0.014925373134328358,...",63.421263,10.386516
4,4,"[63.43337733865644, 10.401074857528158]",False,18,8,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.8, 1.0, 0.6,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.4000000000000001,...","[[0.0, 0.0, 0.0, 0.0, 0.2, 2.2, 0.8, 1.2, 0.2,...","[[0.0, 0.0, 0.0, 0.0, 0.4000000000000001, 2.31...","[[[0.014925373134328358, 0.014925373134328358,...",63.433377,10.401075


## Visualize all stations on a map

In [137]:


# Create a map centered around the average location
map_center = [df_stations['latitude'].mean(), df_stations['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=12)

# Add station markers to the map
for _, row in df_stations.iterrows():
    # 1. Add the standard marker for the location point
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"ID: {row['id']}\nCapacity: {row['capacity']}"
    ).add_to(m)
    
    # 2. Add a 'DivIcon' to display the ID as permanent text
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=DivIcon(
            icon_size=(150,36),
            icon_anchor=(7,20), # Adjust these offsets to position the text
            html=f'<div style="font-size: 10pt; color: black; font-weight: bold;">{row["id"]}</div>',
        )
    ).add_to(m)

# Display the map
m

## Cluster stations on location

In [ ]:
# Encoder to handle numpy types in JSON
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super(NpEncoder, self).default(obj)

# 1. Haversine function to calculate distance between hubs in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# 1. Define City Center (Trondheim Sentrum approx)
city_center = np.array([63.4305, 10.3951])

# 2. Calculate "Remoteness Factor" for each station
# We'll use distance from center as a proxy for 'perifery'
coords = df_stations[['latitude', 'longitude']].values
dist_from_center = np.linalg.norm(coords - city_center, axis=1)

# Normalize scaling factor (0.5 at center, 2.0 at periphery)
# Adjust these values to control how much the clusters "stretch"
scaling_factor = 0.9 + (dist_from_center / dist_from_center.max()) * 2

# 3. Create a Precomputed Scaled Distance Matrix
# Physical Distance
base_dist_matrix = squareform(pdist(coords))

# Scaled Distance: we divide the distance by the average popularity/centrality 
# of the two points. This makes peripheral points "feel" closer to each other.
scaled_dist_matrix = np.zeros_like(base_dist_matrix)
for i in range(len(df_stations)):
    for j in range(len(df_stations)):
        avg_scale = (scaling_factor[i] + scaling_factor[j]) / 2
        # Scaling the threshold: distance / scale
        scaled_dist_matrix[i, j] = base_dist_matrix[i, j] / avg_scale

# 4. Run DBSCAN on the precomputed matrix
# Since we scaled the matrix, a single eps now represents a variable physical distance
db = DBSCAN(eps=0.003, min_samples=2, metric='precomputed').fit(scaled_dist_matrix)

df_stations['cluster'] = db.labels_
# Keep noise (-1) as individual clusters
max_cluster = df_stations['cluster'].max()
noise_mask = df_stations['cluster'] == -1
df_stations.loc[noise_mask, 'cluster'] = range(max_cluster + 1, max_cluster + 1 + noise_mask.sum())

# Mapping setup
id_to_cluster = dict(zip(df_stations['id'], df_stations['cluster']))
unique_clusters = sorted(df_stations['cluster'].unique())
num_clusters = len(unique_clusters)
cluster_id_to_idx = {cid: i for i, cid in enumerate(unique_clusters)}

# 3. Aggregate Station Data
new_stations_list = []
for cluster_id in unique_clusters:
    cluster_stations = df_stations[df_stations['cluster'] == cluster_id]
    
    def aggregate_intensities(series):
        return np.array(series.tolist()).mean(axis=0)

    # 3D Probability Aggregation (Day x Hour x Destination)
    all_cluster_probs = []
    for _, row in cluster_stations.iterrows():
        old_prob_matrix = np.array(row['move_probabilities'])
        new_prob_matrix = np.zeros((7, 24, num_clusters))
        for original_idx, old_id in enumerate(df_stations['id']):
            target_idx = cluster_id_to_idx[id_to_cluster[old_id]]
            new_prob_matrix[:, :, target_idx] += old_prob_matrix[:, :, original_idx]
        all_cluster_probs.append(new_prob_matrix)
    
    final_move_probs = np.mean(all_cluster_probs, axis=0)
    row_sums = final_move_probs.sum(axis=2, keepdims=True)
    final_move_probs = np.divide(final_move_probs, row_sums, out=np.zeros_like(final_move_probs), where=row_sums!=0)

    new_stations_list.append({
        "id": int(cluster_id),
        "merged_ids": cluster_stations['id'].tolist(),
        "location": [float(cluster_stations['latitude'].mean()), float(cluster_stations['longitude'].mean())],
        "is_depot": bool(any(cluster_stations['is_depot'])),
        "capacity": int(cluster_stations['capacity'].sum()),
        "num_bikes": int(cluster_stations['num_bikes'].sum()),
        "leave_intensities": aggregate_intensities(cluster_stations['leave_intensities']),
        "leave_intensities_stdev": aggregate_intensities(cluster_stations['leave_intensities_stdev']),
        "arrive_intensities": aggregate_intensities(cluster_stations['arrive_intensities']),
        "arrive_intensities_stdev": aggregate_intensities(cluster_stations['arrive_intensities_stdev']),
        "move_probabilities": final_move_probs
    })

# 4. Aggregate Global Travel Time Matrices
def aggregate_matrix(old_matrix):
    old_matrix = np.array(old_matrix)
    new_matrix = np.zeros((num_clusters, num_clusters))
    for i, c_i in enumerate(unique_clusters):
        stations_i = df_stations[df_stations['cluster'] == c_i].index.tolist()
        for j, c_j in enumerate(unique_clusters):
            stations_j = df_stations[df_stations['cluster'] == c_j].index.tolist()
            # Mean travel time between all pairs in Cluster i and Cluster j
            submatrix = old_matrix[np.ix_(stations_i, stations_j)]
            new_matrix[i, j] = np.mean(submatrix)
    return new_matrix

traveltime_new = aggregate_matrix(data['traveltime'])
traveltime_stdev_new = aggregate_matrix(data['traveltime_stdev'])
traveltime_vehicle_new = aggregate_matrix(data['traveltime_vehicle'])

# 5. Assemble final structure matching original JSON
output_data = {
    "name": "TD_W34_Clustered",
    "map": data.get("map", "trondheim.png"),
    "map_boundingbox": data.get("map_boundingbox", []),
    "city": data.get("city", "Trondheim"),
    "stations": new_stations_list,
     "bike_class": data.get("bike_class", "Bike"),
    "traveltime": traveltime_new,
    "traveltime_stdev": traveltime_stdev_new,
    "traveltime_vehicle": traveltime_vehicle_new,
    "traveltime_vehicle_stdev": data.get("traveltime_vehicle_stdev", None)
}

# 6. Save versions
with open('clustered_stations.json', 'w') as f:
    json.dump(output_data, f, indent=4, cls=NpEncoder)

with gzip.open('clustered_stations.json.gz', 'wt', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, cls=NpEncoder)

print(f"Clustering complete. {num_clusters} hubs created with all original travel properties preserved.")

Clustering complete. 27 hubs created with all original travel properties preserved.


## Create new map showing clusters

In [139]:
import folium
import json
import pandas as pd
from folium.features import DivIcon
from scipy.spatial import ConvexHull
import numpy as np

# 1. Load Original and Clustered Data
with open('C:\\Users\\Minamsj\\FOMOsim\\policies\\sjovik_sund\\generated_instances\\TD_W34_67.json', 'r') as f:
    orig_data = json.load(f)
df_orig = pd.DataFrame(orig_data['stations'])
df_orig['lat'] = df_orig['location'].apply(lambda x: x[0])
df_orig['lon'] = df_orig['location'].apply(lambda x: x[1])

with open('C:\\Users\\Minamsj\\FOMOsim\\policies\\sjovik_sund\\scripts\\test_instance\\clustered_stations.json', 'r') as f:
    clustered_data = json.load(f)
hubs = clustered_data['stations']

# 2. Setup Map center
map_center = [df_orig['lat'].mean(), df_orig['lon'].mean()]
m = folium.Map(location=map_center, zoom_start=13, tiles='cartodbpositron')

# Color palette for clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 
          'darkpurple', 'pink', 'darkblue', 'gray', 'black']

# 3. Iterate through Hubs to draw Shadows and Markers
for i, hub in enumerate(hubs):
    hub_id = hub['id']
    merged_ids = hub['merged_ids']
    color = colors[i % len(colors)]
    
    # Filter original stations belonging to this specific hub
    stations_in_hub = df_orig[df_orig['id'].isin(merged_ids)]
    coords = stations_in_hub[['lat', 'lon']].values
    
    # --- DRAW SHADOW (Convex Hull) ---
    if len(coords) >= 3:
        try:
            hull = ConvexHull(coords)
            hull_points = coords[hull.vertices].tolist()
            folium.Polygon(
                locations=hull_points,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.15, # Light shadow effect
                weight=1,
                popup=f"Hub {hub_id}"
            ).add_to(m)
        except:
            # Fallback for collinear points
            folium.PolyLine(locations=coords.tolist(), color=color, weight=8, opacity=0.2).add_to(m)
    elif len(coords) == 2:
        # Simple line shadow for 2-station clusters
        folium.PolyLine(locations=coords.tolist(), color=color, weight=8, opacity=0.2).add_to(m)

    # --- DRAW ORIGINAL STATION MARKERS ---
    for _, row in stations_in_hub.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            popup=f"Station ID: {row['id']}<br>Assigned to Hub: {hub_id}"
        ).add_to(m)
        
        # Display ID Label
        folium.Marker(
            location=[row['lat'], row['lon']],
            icon=DivIcon(
                icon_size=(150,36),
                icon_anchor=(5,18),
                html=f'<div style="font-size: 8pt; color: black; font-weight: bold; text-shadow: 0 0 3px white;">{row["id"]}</div>',
            )
        ).add_to(m)

# Save and view
m.save('cluster_shadow_map.html')